# NMF on ModulAir 00687

In [1]:
from sklearn.decomposition import NMF
import numpy as np
import pandas as pd

## Cleaning Data

In [2]:
#importing data from Modulair MOD-00687
data = pd.read_csv('MOD-00687.csv').set_index('timestamp')
data.head()

,id,timestamp_local,sn,rh,temp,bin0,bin1,bin2,bin3,bin4,...,no2,o3,pm1_model_id,pm25_model_id,pm10_model_id,co_model_id,no_model_id,no2_model_id,o3_model_id,ws_scalar
timestamp,,,,,,,,,,,,,,,,,,,,,
2025-12-27T14:09:49Z,574530654,2025-12-27T09:09:49Z,MOD-00687,76.0,-1.7,3.527,0.691,0.185,0.056,0.043,...,33.079,14.604,14334.0,14335.0,14336.0,14474.0,14499.0,14549.0,14524.0,2.48
2025-12-27T14:08:49Z,574530653,2025-12-27T09:08:49Z,MOD-00687,76.4,-1.8,3.761,0.627,0.218,0.041,0.078,...,32.818,14.558,14334.0,14335.0,14336.0,14474.0,14499.0,14549.0,14524.0,3.41
2025-12-27T14:07:49Z,574530652,2025-12-27T09:07:49Z,MOD-00687,76.5,-1.8,3.794,0.652,0.228,0.058,0.072,...,32.577,14.544,14334.0,14335.0,14336.0,14474.0,14499.0,14549.0,14524.0,5.03
2025-12-27T14:06:49Z,574530651,2025-12-27T09:06:49Z,MOD-00687,76.4,-1.8,4.033,0.634,0.246,0.053,0.042,...,32.581,14.558,14334.0,14335.0,14336.0,14474.0,14499.0,14549.0,14524.0,2.93
2025-12-27T14:05:49Z,574528724,2025-12-27T09:05:49Z,MOD-00687,77.4,-1.8,4.031,0.639,0.195,0.070,0.060,...,33.013,13.612,14334.0,14335.0,14336.0,14474.0,14499.0,14549.0,14524.0,4.50


In [3]:
#only columns of interest
COLS_TO_INCLUDE = ['timestamp_local','co','no','no2','o3','bin0','bin1','bin2','bin3','bin4','bin5']
data = data[COLS_TO_INCLUDE]

data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
timestamp,,,,,,,,,,,
2025-12-27T14:09:49Z,2025-12-27T09:09:49Z,783.067,1.723,33.079,14.604,3.527,0.691,0.185,0.056,0.043,0.052
2025-12-27T14:08:49Z,2025-12-27T09:08:49Z,781.854,1.919,32.818,14.558,3.761,0.627,0.218,0.041,0.078,0.083
2025-12-27T14:07:49Z,2025-12-27T09:07:49Z,792.866,1.919,32.577,14.544,3.794,0.652,0.228,0.058,0.072,0.031
2025-12-27T14:06:49Z,2025-12-27T09:06:49Z,783.496,1.919,32.581,14.558,4.033,0.634,0.246,0.053,0.042,0.048
2025-12-27T14:05:49Z,2025-12-27T09:05:49Z,812.342,1.976,33.013,13.612,4.031,0.639,0.195,0.070,0.060,0.040


In [4]:
#removing the UTC time
data = data.reset_index(drop = True)
data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-12-27T09:09:49Z,783.067,1.723,33.079,14.604,3.527,0.691,0.185,0.056,0.043,0.052
1,2025-12-27T09:08:49Z,781.854,1.919,32.818,14.558,3.761,0.627,0.218,0.041,0.078,0.083
2,2025-12-27T09:07:49Z,792.866,1.919,32.577,14.544,3.794,0.652,0.228,0.058,0.072,0.031
3,2025-12-27T09:06:49Z,783.496,1.919,32.581,14.558,4.033,0.634,0.246,0.053,0.042,0.048
4,2025-12-27T09:05:49Z,812.342,1.976,33.013,13.612,4.031,0.639,0.195,0.070,0.060,0.040


In [5]:
#converting to datetime
data['timestamp_local'] = pd.to_datetime(data['timestamp_local'],
                                       format='%Y-%m-%dT%H:%M:%SZ',
                                       exact=False)
data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-12-27 09:09:49,783.067,1.723,33.079,14.604,3.527,0.691,0.185,0.056,0.043,0.052
1,2025-12-27 09:08:49,781.854,1.919,32.818,14.558,3.761,0.627,0.218,0.041,0.078,0.083
2,2025-12-27 09:07:49,792.866,1.919,32.577,14.544,3.794,0.652,0.228,0.058,0.072,0.031
3,2025-12-27 09:06:49,783.496,1.919,32.581,14.558,4.033,0.634,0.246,0.053,0.042,0.048
4,2025-12-27 09:05:49,812.342,1.976,33.013,13.612,4.031,0.639,0.195,0.070,0.060,0.040


In [6]:
#taking hourly average of df. round to floor of the hour
data = data.groupby(data['timestamp_local'].dt.floor('h')).agg(co = ('co','mean'),
                                                         no2 = ('no2','mean'),
                                                         o3 = ('o3','mean'),
                                                         no = ('no','mean'),
                                                         bin0 = ('bin0','mean'),
                                                         bin1 = ('bin1','mean'),
                                                         bin2 = ('bin2','mean'),
                                                         bin3 = ('bin3','mean'),
                                                         bin4 = ('bin4','mean'),
                                                         bin5 = ('bin5','mean')).reset_index()

data = data.round(decimals = 2)
data = data.dropna()
data

,timestamp_local,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-03-31 20:00:00,906.87,34.52,20.42,1.82,19.92,2.61,1.35,0.49,0.67,0.55
1,2025-03-31 21:00:00,1011.23,37.81,17.82,2.49,29.94,5.66,2.05,0.63,0.86,0.75
2,2025-03-31 22:00:00,1005.31,35.44,24.59,2.76,26.19,7.89,2.40,0.60,0.72,0.61
3,2025-03-31 23:00:00,807.25,23.98,34.32,2.78,17.64,6.75,2.15,0.43,0.40,0.24
4,2025-04-01 00:00:00,756.74,24.93,40.28,2.56,15.53,3.84,1.18,0.26,0.27,0.18
...,...,...,...,...,...,...,...,...,...,...,...
6229,2025-12-27 05:00:00,816.94,33.34,11.88,2.73,3.39,0.57,0.20,0.06,0.06,0.05
6230,2025-12-27 06:00:00,801.46,33.23,12.18,2.50,3.68,0.58,0.21,0.06,0.05,0.04
6231,2025-12-27 07:00:00,801.97,32.72,12.00,2.27,4.57,0.77,0.28,0.07,0.08,0.07
6232,2025-12-27 08:00:00,761.54,32.62,13.74,1.99,3.83,0.65,0.22,0.06,0.06,0.05


In [7]:
#setting local time as index
data = data.set_index('timestamp_local')
data.head()

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-03-31 20:00:00,906.87,34.52,20.42,1.82,19.92,2.61,1.35,0.49,0.67,0.55
2025-03-31 21:00:00,1011.23,37.81,17.82,2.49,29.94,5.66,2.05,0.63,0.86,0.75
2025-03-31 22:00:00,1005.31,35.44,24.59,2.76,26.19,7.89,2.40,0.60,0.72,0.61
2025-03-31 23:00:00,807.25,23.98,34.32,2.78,17.64,6.75,2.15,0.43,0.40,0.24
2025-04-01 00:00:00,756.74,24.93,40.28,2.56,15.53,3.84,1.18,0.26,0.27,0.18


In [8]:
#subsetting for only positive and non NA values
data = data.where(data>0)
data = data.dropna()

In [9]:
#scaling
def maximum_absolute_scaling(df):
    # copy the dataframe
    df_scaled = df.copy()
    # apply maximum absolute scaling
    for column in df_scaled.columns:
        df_scaled[column] = df_scaled[column]  / df_scaled[column].abs().max()
    return df_scaled

# call the maximum_absolute_scaling function
data = maximum_absolute_scaling(data)

In [10]:
#subsetting for only positive and non NA values
data = data.where(data>0)
data = data.dropna()
data.head()

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-03-31 20:00:00,0.474141,0.691645,0.239953,0.051066,0.233556,0.084933,0.075503,0.094412,0.149554,0.314286
2025-03-31 21:00:00,0.528703,0.757564,0.209401,0.069865,0.351038,0.184185,0.114653,0.121387,0.191964,0.428571
2025-03-31 22:00:00,0.525608,0.710078,0.288954,0.077441,0.307070,0.256752,0.134228,0.115607,0.160714,0.348571
2025-03-31 23:00:00,0.422056,0.480465,0.403290,0.078002,0.206824,0.219655,0.120246,0.082852,0.089286,0.137143
2025-04-01 00:00:00,0.395648,0.499499,0.473325,0.071829,0.182085,0.124959,0.065996,0.050096,0.060268,0.102857


In [11]:
data.to_csv(r'MOD-00687_timeseries_hourly_scaled.csv')

In [12]:
# Check start date of df to make sure that it's loaded in correctly
start = data.index.min()

end = data.index.min()

print(start, end)

2025-03-31 20:00:00 2025-03-31 20:00:00


## Implementing NMF

In [13]:
#setting up the NMF
nmf = NMF(n_components = 4, max_iter = 8000)

In [14]:
W = nmf.fit_transform(data)
H = nmf.components_
V = pd.DataFrame(np.dot(W,H), columns=data.columns)
V.index = data.index
V

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-03-31 20:00:00,0.520925,0.678608,0.220598,0.076023,0.253566,0.156291,0.076671,0.055484,0.063061,0.103870
2025-03-31 21:00:00,0.591732,0.741086,0.183372,0.083852,0.386224,0.252488,0.123862,0.088179,0.095083,0.146394
2025-03-31 22:00:00,0.570123,0.699756,0.263410,0.082311,0.386023,0.245579,0.120473,0.085371,0.090635,0.136569
2025-03-31 23:00:00,0.431755,0.480041,0.390199,0.065739,0.284400,0.163072,0.079998,0.056394,0.058810,0.086355
2025-04-01 00:00:00,0.405429,0.497915,0.466099,0.064781,0.214676,0.108636,0.053293,0.037633,0.039476,0.058467
...,...,...,...,...,...,...,...,...,...,...
2025-12-27 05:00:00,0.424433,0.669241,0.141198,0.063836,0.039900,0.014574,0.007149,0.007743,0.017872,0.047480
2025-12-27 06:00:00,0.414846,0.667158,0.145140,0.062686,0.041467,0.015755,0.007729,0.007983,0.017513,0.045633
2025-12-27 07:00:00,0.417270,0.656114,0.141716,0.062689,0.053546,0.024264,0.011903,0.010883,0.020384,0.049514


In [15]:
W_df = pd.DataFrame(W, columns =[f'Feature {i+1}' for i in range(0,4)]) #array-like of shape (n_samples, n_components)
W_df

,Feature 1,Feature 2,Feature 3,Feature 4
0,0.051948,0.025289,0.080501,0.120890
1,0.054926,0.017223,0.130050,0.150231
2,0.051908,0.033470,0.126491,0.133871
3,0.035959,0.062818,0.083994,0.079777
4,0.038933,0.076337,0.055955,0.055122
...,...,...,...,...
6085,0.054254,0.009394,0.007507,0.090836
6086,0.054156,0.010178,0.008115,0.086208
6087,0.053016,0.009893,0.012498,0.089056
6088,0.053244,0.013728,0.009269,0.076180


In [16]:
COLS_TO_INCLUDE.pop(0)
COLS_TO_INCLUDE

['co', 'no', 'no2', 'o3', 'bin0', 'bin1', 'bin2', 'bin3', 'bin4', 'bin5']

In [17]:
H_df = pd.DataFrame(H, index = [f'Feature {i+1}' for i in range(0,4)], columns = COLS_TO_INCLUDE) #array-like of shape (n_components, n_features)
H_df

,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
Feature 1,3.667771,11.858687,1.695018,0.639459,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Feature 2,1.691172,0.000000,5.241306,0.314279,0.723003,0.000000,0.000000,0.000000,0.000000,0.000000
Feature 3,0.123873,0.399249,0.000000,0.000000,2.711911,1.941476,0.952424,0.640743,0.557016,0.576934
Feature 4,2.296733,0.251733,0.000000,0.288334,0.140372,0.000000,0.000000,0.032289,0.150720,0.475026


In [18]:
#converting the results to a dataframe
results = pd.DataFrame(W,index=data.index) #H is time series data of the factors, W is the comp (coeff matrix)
results.columns = ["Factor {}".format(i+1) for i in range(H.T.shape[1])]
results

,Factor 1,Factor 2,Factor 3,Factor 4
timestamp_local,,,,
2025-03-31 20:00:00,0.051948,0.025289,0.080501,0.120890
2025-03-31 21:00:00,0.054926,0.017223,0.130050,0.150231
2025-03-31 22:00:00,0.051908,0.033470,0.126491,0.133871
2025-03-31 23:00:00,0.035959,0.062818,0.083994,0.079777
2025-04-01 00:00:00,0.038933,0.076337,0.055955,0.055122
...,...,...,...,...
2025-12-27 05:00:00,0.054254,0.009394,0.007507,0.090836
2025-12-27 06:00:00,0.054156,0.010178,0.008115,0.086208
2025-12-27 07:00:00,0.053016,0.009893,0.012498,0.089056


In [19]:
COLS_TO_INCLUDE = ['co','no','no2','o3','bin0','bin1','bin2','bin3','bin4','bin5']
comp = pd.DataFrame(H, index = results.columns, columns = COLS_TO_INCLUDE)
comp

,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
Factor 1,3.667771,11.858687,1.695018,0.639459,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Factor 2,1.691172,0.000000,5.241306,0.314279,0.723003,0.000000,0.000000,0.000000,0.000000,0.000000
Factor 3,0.123873,0.399249,0.000000,0.000000,2.711911,1.941476,0.952424,0.640743,0.557016,0.576934
Factor 4,2.296733,0.251733,0.000000,0.288334,0.140372,0.000000,0.000000,0.032289,0.150720,0.475026


In [20]:
res = []

for col in comp.columns:
    #for each factor, compute its contribution to the species in V
    by_factor = pd.Series(dtype=float)
    for i, factor in enumerate(results.columns):
        contrib = H[i, data.columns.get_loc(col)] * W[:, i].sum()
        by_factor.at[factor] = contrib

    #normalizing by the total amount of that species in the original data
    by_factor /= data[col].sum()

    #adding as a row to the resulting dataframe
    res.append(pd.DataFrame(by_factor).T.rename(index={0: col}))

res = pd.concat(res)
res.columns = results.columns
res['Residual'] = 1 - res.sum(axis=1)

res

,Factor 1,Factor 2,Factor 3,Factor 4,Residual
co,0.378088,0.211419,0.008962,0.401479,0.000051
no,0.420022,0.250346,0.000000,0.321156,0.008476
no2,0.943820,0.000000,0.022301,0.033975,-0.000096
o3,0.210521,0.789454,0.000000,0.000000,0.000025
bin0,0.000000,0.295142,0.640671,0.080125,-0.015938
bin1,0.000000,0.000000,1.056837,0.000000,-0.056837
bin2,0.000000,0.000000,1.065198,0.000000,-0.065198
bin3,0.000000,0.000000,0.928484,0.113051,-0.041535
bin4,0.000000,0.000000,0.615922,0.402675,-0.018597
bin5,0.000000,0.000000,0.338091,0.672591,-0.010682


In [21]:
results.to_csv(r'MOD-00687_4_factor_results.csv')
comp.to_csv(r'MOD-00687_4_factor_comp.csv')
res.to_csv(r'MOD-00687_4_factor_resid.csv')

In [22]:
# check how many rows of data you have
len(data)

6090